# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.co

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'capabilities page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'capabilities page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'blog', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'external company',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'social profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'social profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'social profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 10 relevant links


{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'services page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'services page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'projects page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'projects page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [11]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 3 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'company page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [14]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [13]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 18 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Qwen/Qwen3.5-35B-A3B
Updated
1 day ago
•
378k
•
675
Qwen/Qwen3.5-27B
Updated
4 days ago
•
172k
•
434
unsloth/Qwen3.5-35B-A3B-GGUF
Updated
about 23 hours ago
•
350k
•
356
Qwen/Qwen3.5-122B-A10B
Updated
4 days ago
•
120k
•
344
Qwen/Qwen3.5-397B-A17B
Updated
5 days ago
•
889k
•
1.13k
Browse 2M+ models
Spaces
Running
on
Zero
Featured
1.74k
Qwen Image Multiple Angles 3D Camera
🎥
1.74k
Change the camera angle of a photo with AI
Running
on
Zero
Featured
230
Omni Video Factory
🏆
230
text to video, image to video, video extend
Running
on
Zero

In [15]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [16]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [17]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 20 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nQwen/Qwen3.5-35B-A3B\nUpdated\n1 day ago\n•\n378k\n•\n675\nQwen/Qwen3.5-27B\nUpdated\n4 days ago\n•\n172k\n•\n434\nunsloth/Qwen3.5-35B-A3B-GGUF\nUpdated\nabout 23 hours ago\n•\n350k\n•\n356\nQwen/Qwen3.5-122B-A10B\nUpdated\n4 days ago\n•\n120k\n•\n344\nQwen/Qwen3.5-397B-A17B\nUpdated\n5 days ago\n•\n889k\n•\n1.13k\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nFeatured\n1.74k\nQwen Image Mu

In [18]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [19]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


# Hugging Face Brochure

## About Hugging Face  
Hugging Face is the leading collaboration platform for the machine learning (ML) community, dedicated to building the future of AI through open and ethical innovation. It serves as a central hub where ML engineers, scientists, and enthusiasts can share, explore, and experiment with over 2 million open-source models, datasets, and applications spanning text, image, video, audio, and 3D modalities.

With a fast-growing global community, Hugging Face powers some of the most widely used open-source ML libraries and tools. It fosters knowledge sharing and innovation by enabling users to create portfolios, collaborate on cutting-edge research, and build AI applications that push the technology frontier.

---

## What Hugging Face Offers

### The Hub for Machine Learning  
- Host unlimited public models, datasets, and apps.  
- Discover and collaborate across millions of resources.  
- Share work and build your ML profile on a trusted global platform.

### Multi-Modal AI  
Explore and build with text, images, video, audio, and even 3D data, enabling diverse AI applications from speech synthesis to complex 3D computer vision.

### Open Source & Community  
- Contribute to and benefit from the largest open ML ecosystem.  
- Join a passionate community driven by collaborative progress and ethical AI development.

### Enterprise Solutions  
Hugging Face supports organizations with scaling AI through:  
- Enterprise-grade security, Single Sign-On (SSO), and compliance.  
- Granular access control with resource groups and token management.  
- Advanced analytics to monitor and optimize usage.  
- Flexible contracts and additional compute power options (e.g., ZeroGPU Quota Boost).  
- Private dataset viewers and extended private storage for enhanced team collaboration and data protection.

---

## Customers and Community  
Hugging Face’s platform is trusted by a wide range of ML practitioners—from independent researchers and startups to large enterprises seeking to accelerate AI innovation securely. The thriving community actively maintains and updates popular models like Qwen series (Qwen3.5 and related variants) and contributes datasets, applications, and demos that reflect the cutting edge of AI research and deployment.

---

## Company Culture and Careers  
Hugging Face is a mission-driven organization at the heart of the AI revolution. The culture is built around openness, collaboration, ethical AI, and continuous learning. Employees and contributors are empowered to work on challenging problems, share knowledge openly, and directly impact the future of machine learning.

Careers at Hugging Face offer opportunities to join a highly talented team advancing the state of AI, contribute to influential open source projects, and help shape tools and platforms used worldwide by AI professionals.

Explore current opportunities on their website and be part of a diverse, innovative, and inclusive work environment passionately building the future of AI.

---

## Get Involved  
- Join the global Hugging Face community by signing up on their platform.  
- Browse models, datasets, and applications.  
- Build and share AI-powered projects via Spaces.  
- Access enterprise-grade tools to securely scale AI in your organization.  

**Colors & Branding:**  
- Signature colors: #FFD21E (yellow), #FF9D00 (orange), #6B7280 (gray)  
- Brand assets available for contributors and partners to maintain consistent, vibrant identity.

---

## Connect and Learn More  
- Website: https://huggingface.co  
- Documentation & Forums for developers and beginners alike.  
- Active social communities on GitHub, Twitter, LinkedIn, and Discord.  

**Hugging Face — Where Machine Learning Meets Collaboration and Innovation.**

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [20]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [21]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


# Hugging Face: The AI Community Building the Future

---

## About Hugging Face

Hugging Face is a leading collaboration platform for the machine learning (ML) community, dedicated to building an open and ethical AI future. Serving as a central hub, Hugging Face empowers ML engineers, scientists, and end users to create, explore, discover, and share open-source models, datasets, and applications.

Their platform fosters collaboration and innovation by enabling users to host unlimited public ML models, datasets, and applications across a wide range of data modalities including text, images, video, audio, and even 3D.

---

## What They Offer

- **Extensive Model Repository**: Access over 2 million pre-trained models, including trending state-of-the-art ones like Qwen/Qwen3.5 variants, covering a variety of AI tasks.
- **Datasets Library**: Browse from more than 500,000 datasets updated regularly to support cutting-edge research and development.
- **Spaces**: Explore and run more than 1 million AI applications built and shared by the community, such as AI-powered 3D camera angle changers, video generation tools, and speech synthesis demos.
- **Open-Source Stack**: Leverage the Hugging Face open-source tools to accelerate your machine learning workflows and research.
- **Enterprise Solutions**: Access paid compute resources and enterprise-grade services for scaling AI applications.

---

## Community and Culture

At its core, Hugging Face is a vibrant, fast-growing community passionate about open-source machine learning. The company emphasizes:

- **Collaboration**: Encouraging collaboration among ML practitioners worldwide.
- **Ethical AI**: Committing to building AI responsibly, prioritizing transparency, inclusivity, and accessibility.
- **Learning and Growth**: Supporting users in building portfolios by sharing their work and contributing to the AI ecosystem.
- **Diversity of Modalities**: Innovating across multiple data types—text, audio, video, imagery, and 3D—to foster versatile AI advancements.

---

## Customers and Users

Hugging Face's platform attracts a diverse audience:

- **Machine Learning Engineers & Researchers**: Who share and experiment with state-of-the-art models and datasets.
- **Data Scientists and Developers**: Building robust AI applications using pre-trained models or creating custom solutions.
- **Enterprises**: Utilizing scalable and secure AI tools, enhanced with Hugging Face’s enterprise compute capabilities.
- **Educators & Students**: Benefiting from open access to ML resources to foster next-generation AI expertise.

---

## Careers and Opportunities

Hugging Face is continuously expanding and seeks passionate professionals to join their mission of democratizing AI.

- **Innovative environment**: Be part of a pioneering AI company pushing the boundaries of open-source machine learning.
- **Community-driven culture**: Collaborate with a global network of talented engineers and researchers.
- **Growth**: Build and showcase your ML expertise within a supportive and dynamic ecosystem.

Keep an eye on their careers page and community channels for opportunities to contribute and grow within the AI frontier.

---

## Brand & Visual Identity

Hugging Face has a distinctive and friendly brand presence, with vibrant colors like bright yellow (#FFD21E) and orange (#FF9D00), reflecting creativity and energy, supported by modern grays. The brand assets and logos are openly available for collaboration and community use.

---

## Join Hugging Face Today!

- Collaborate on state-of-the-art machine learning models and datasets
- Explore an expansive ecosystem of AI applications
- Accelerate your ML projects with open-source tools and enterprise solutions
- Be part of a global movement shaping the future of AI

**Visit:** [huggingface.co](https://huggingface.co)  
**Sign Up:** Build your AI portfolio and join the community of innovators!

---

*Hugging Face – The AI community building the future.*

In [23]:
create_brochure("Peter Strössler's online portfolio", "https://peterstroessler.com")

Selecting relevant links for https://peterstroessler.com by calling gpt-5-nano
Found 8 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


# Peter Strössler - PSWeb  
### Professioneller Software Engineer mit Herz und Seele

---

## Über Peter Strössler

Peter Strössler ist ein erfahrener und leidenschaftlicher Software Engineer, dessen Beruf seine Passion ist. Er begann 2006 seine Karriere mit Web Publisher SIZ und der Programmierung von CMS in PHP, HTML und CSS. Seine fundierten Informatikkenntnisse hat er während seiner Lehre an der WISS erweitert und durch ein Studium als Informatik-Ingenieur an der ZHAW Winterthur vertieft.

Peter lebt nach dem Motto "Ich weiss, dass ich nichts weiss" und bildet sich kontinuierlich weiter, um stets aktuelle Technologien und Best Practices anzuwenden.

---

## Dienstleistungen

Peter bietet ein breites Spektrum an Services rund um die Softwareentwicklung:

- **Frontend Entwicklung**  
  Kompetent in HTML5, CSS und Typescript sowie erfahrener Umgang mit ReactJS.

- **Backend Entwicklung**  
  Erfahrung mit NodeJs, Java (Springboot) und PHP (Codeigniter) zur Erstellung robuster Webservices.

- **Graphic Design**  
  Erstellung und Aufbereitung von Grafiken für das Web mit der nötigen Expertise.

- **Consulting**  
  Beratung und Unterstützung bei Fragen und Projekten rund um Webtechnologien.

---

## Projekte & Referenzen

Peter hat vielseitige Projekte realisiert, die von Web-Design über AI-Anwendungen bis hin zu mobilen Apps reichen. Eine Auswahl seiner Arbeiten umfasst:

- Währungsrechner (App)
- Wordpress-Seite für Fian Schweiz
- KI Projekte wie Katzen- und Hundeerkennung
- Diverse React/Typescript Apps (Todo App, Connect4, ReactJS Webapp mit AI)
- Angular Hotel App
- PWA/Websocket Schachapplikation
- Backend Projekte mit NodeJs, MongoDB auf ARM Cloudservern
- AI-basierte Lösungen wie Diagnose Speech2Structure und AI Shirt Replacement
- KI-Projekte mit künstlichen neuronalen Netzen (ANN Regression Numeric & Classification)
- Server-Implementierung mit Flask und Llama 3.1 LLM
- Kreative Arbeiten wie AI generierte Spionageromane

Viele seiner Projekte sind auf seinem [Github Account](#) verfügbar, und seine Webprojekte aus dem Bachelor-Studium können eingesehen werden.

---

## Unternehmenskultur

Peter Strössler arbeitet mit einer persönlichen und engagierten Haltung. Er sieht sich als lebenslangen Lerner und bringt viel Herzblut in jedes seiner Projekte. Seine Arbeit ist von Leidenschaft und tiefer technischer Kompetenz geprägt. Offenheit für neue Technologien und eine lösungsorientierte Denkweise stehen im Mittelpunkt seines Schaffens.

---

## Kunden

Peter richtet sich sowohl an Privatpersonen als auch Unternehmen, die professionelle Web- und Softwarelösungen benötigen. Seine breite Expertise ermöglicht es ihm, individuelle Projekte verschiedenster Branchen abzudecken — von Startups bis hin zu etablierten Firmen.

---

## Karriere & Kontakt

Derzeit handelt es sich um das persönliche Portfolio von Peter Strössler als Einzelperson. Er freut sich über interessante Projektanfragen oder Kooperationen.  
**Kontakt:** Sie können Peter jederzeit per Email erreichen.

**CV Download:** Peter stellt seinen Lebenslauf zum Download bereit, um seine Erfahrungen und Qualifikationen transparent zu machen.

---

### Zusammenfassung

Peter Strössler bietet professionelle Softwareentwicklung mit Leidenschaft und tiefgehendem Know-how. Sein Portfolio zeigt eine breite Palette von Web-, Mobile- und KI-Projekten. Wer auf der Suche nach einem zuverlässigen, engagierten Software Engineer mit kreativen Ideen und fundierten technischen Fähigkeiten ist, findet mit Peter einen idealen Partner.

---

*Kontaktieren Sie Peter Strössler für Ihr nächstes Softwareprojekt und profitieren Sie von einem erfahrenen Profi, der Technik und kreative Lösungen vereint.*

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>